[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C71_Prompt_Programming_Course/05_prompt_ops/05_prompt_ops.ipynb)

# 05 · 提示的运维与跨模型迁移（失效 / 迁移 / 成本 / 灰度 / 监控 / 运维卡）

目标：把 prompt 当成一个**会随模型失效的资产**来管理。

本 notebook 你会亲手实现：
1. **四个「模型」的模拟器** —— 它们在近因偏置、格式依从度、标签先验上不同
2. **同一个 prompt 的跨模型分数跨度** —— 以及**为 A 优化的 prompt 在 B 上比基线更差**
3. **迁移的分层复核** —— 总分看不出来的那一层
4. **成本模型** —— token 数 × 单价 × (1 − 缓存命中率)，含一个「单价更低但总成本更高」的例子
5. **灰度分桶** —— 某一层塌了而总分完全没动
6. **监控四项** —— 修复率是解析率的前导指标
7. **prompt 运维卡 + 门禁**

> 心智模型：**签名、取值域、格式、解码约束跨模型保留；
> 示例顺序、指令措辞、校准参数不保留。把它们分开存，换模型就只换那一半。**

## 0 · 环境与四个「模型」

In [ ]:
import os, re, json, math, hashlib, itertools
from collections import Counter, defaultdict

import numpy as np

DIM = 2048
LABELS = ['bug', 'feature', 'billing', 'account', 'other']
UNPARSEABLE = 'UNPARSEABLE'

def tokenize(text):
    text = text.lower()
    toks = re.findall(r'[a-z0-9]+', text)
    for run in re.findall(r'[\u4e00-\u9fff]+', text):
        toks += list(run)
        toks += [run[i:i + 2] for i in range(len(run) - 1)]
    return toks

def embed(text, dim=DIM):
    v = np.zeros(dim)
    for tok in tokenize(text):
        v[int(hashlib.md5(tok.encode()).hexdigest(), 16) % dim] += 1.0
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

# 四个「模型」：在三个行为维度上不同
MODELS = {
    'model-a@2026-01': dict(recency=+0.35, prior='other', format_compliance=0.98,
                            price_in=1.0, token_ratio=1.00),
    'model-a@2026-06': dict(recency=+0.20, prior='other', format_compliance=0.95,
                            price_in=1.0, token_ratio=1.00),   # 同家族小版本
    'model-b@2026-06': dict(recency=-0.30, prior='billing', format_compliance=0.90,
                            price_in=0.6, token_ratio=1.30),   # 另一家族：近因偏置**反向**
    'model-c@2026-06': dict(recency=+0.05, prior='bug', format_compliance=0.99,
                            price_in=2.0, token_ratio=0.85),
}

def toy_lm(model_id, instruction, demos, x, allowed=None):
    """同一套接口，行为由 model_id 决定。

    recency 为负 = **首因偏置**（靠前的示例更重）——真实模型的位置偏置方向确实不一致。
    """
    cfg = MODELS[model_id]
    allowed = LABELS if allowed is None else allowed
    declared = [l for l in allowed if l in instruction]
    if not declared:
        return UNPARSEABLE
    if not demos:
        return cfg['prior']
    E = np.stack([embed(t) for t, _ in demos])
    s = E @ embed(x)
    n = len(demos)
    s = s + np.array([cfg['recency'] * (i / (n - 1) if n > 1 else 1.0)
                      for i in range(n)])
    label = demos[int(np.argmax(s))][1]
    return label if label in declared else cfg['prior']

print('四个模型的行为参数:')
for m, c in MODELS.items():
    print(f'  {m:<18} recency {c["recency"]:+.2f}  prior {c["prior"]:<9}'
          f' format {c["format_compliance"]:.2f}  price {c["price_in"]:.1f}'
          f'  token×{c["token_ratio"]:.2f}')

## 1 · 数据与一个 prompt bundle

**bundle 分成两半**：可迁移的（签名/取值域/格式/解码）与不可迁移的（顺序/措辞/校准）。

In [ ]:
POOL = [
    ('登录后一直转圈，点不动', 'bug'), ('保存文件时报错 500', 'bug'),
    ('页面加载不出来', 'bug'), ('导出 CSV 会丢最后一行', 'bug'),
    ('希望支持批量导出', 'feature'), ('能不能加暗色主题', 'feature'),
    ('想要一个搜索框', 'feature'), ('建议增加导入模板', 'feature'),
    ('这个月扣了两次钱', 'billing'), ('发票开错了公司名', 'billing'),
    ('为什么涨价了', 'billing'), ('想申请退款', 'billing'),
    ('忘记密码收不到邮件', 'account'), ('想改绑定手机号', 'account'),
    ('账号被锁了', 'account'), ('想注销账号', 'account'),
    ('你们客服态度不错', 'other'), ('随便看看', 'other'),
    ('没什么事', 'other'), ('祝好', 'other'),
]
TEST = [
    ('打开报表就崩溃', 'bug'), ('保存草稿会丢内容', 'bug'),
    ('希望能加个批量删除', 'feature'), ('想要导出 PDF 的功能', 'feature'),
    ('这个月账单不对', 'billing'), ('发票上的税号不对', 'billing'),
    ('登录不上，密码重置也收不到', 'account'), ('手机号换了要怎么改', 'account'),
    ('感谢你们的帮助', 'other'), ('没别的了', 'other'),
]

PORTABLE = dict(                      # ← 与模型无关，一份就够
    signature=dict(inputs=['feedback'], output='label', domain=LABELS),
    instruction_core=('把用户反馈分类为 bug / feature / billing / account / other 之一，'
                      '只输出标签。都不符合时输出 other。'),
    format_spec='json {"label": "<标签>"}',
    decoding=dict(constrained=True, grammar='json+enum'),
)
def make_bundle(model_id, demo_order=None, wording_suffix='', calib=None, k=8):
    """不可迁移的部分：示例顺序、措辞后缀、校准。"""
    demos = [POOL[i] for i in (demo_order if demo_order else range(k))]
    return dict(model_id=model_id, demos=demos,
                instruction=PORTABLE['instruction_core'] + wording_suffix,
                calib=calib, **PORTABLE)

def run_bundle(b, test=None, apply_format_noise=True, seed=0):
    """返回 dict(parse_rate, repair_rate, end_to_end, by_label, pred_dist)。"""
    test = TEST if test is None else test
    cfg = MODELS[b['model_id']]
    n_ok = n_rep = n_c = 0
    by_label, preds = defaultdict(list), Counter()
    for x, gold in test:
        raw = toy_lm(b['model_id'], b['instruction'], b['demos'], x)
        parsed = raw if raw in LABELS else None
        repaired = False
        if apply_format_noise and parsed is not None:
            h = int(hashlib.md5((x + b['model_id'] + str(seed)).encode()), 16) \
                if False else int(hashlib.md5((x + b['model_id']).encode()).hexdigest(), 16)
            if (h % 1000) / 1000.0 > cfg['format_compliance']:
                if b['decoding']['constrained']:
                    pass                       # 约束下不可能不合格式
                else:
                    repaired = True            # 语法层修复救回来
        n_rep += repaired
        if parsed is not None:
            n_ok += 1
        hit = (parsed == gold)
        n_c += hit
        by_label[gold].append(float(hit))
        preds[parsed] += 1
    n = len(test)
    return dict(parse_rate=n_ok / n, repair_rate=n_rep / n, end_to_end=n_c / n,
                by_label={l: float(np.mean(v)) for l, v in by_label.items()},
                pred_dist=dict(preds))

B_A = make_bundle('model-a@2026-01')
r = run_bundle(B_A)
print('bundle_A on model-a@2026-01:',
      {k: (round(v, 3) if isinstance(v, float) else v)
       for k, v in r.items() if k not in ('by_label', 'pred_dist')})
assert r['parse_rate'] == 1.0
print('\n✅ PORTABLE 四项（签名/取值域/格式/解码）与模型无关；')
print('   demos 顺序、措辞、校准是每个模型一份。这个拆分是本模块的设计核心。')

## 2 · 跨模型：同一个 prompt 的分数跨度

以及本模块的核心结果：**为 A 优化的 prompt 在 B 上可能比基线更差。**

In [ ]:
# 覆盖全部五个标签的基线示例集（模块 02 第 2 节：不覆盖会让某类归零）
BASELINE_ORDER = [0, 4, 8, 12, 16, 1, 5, 9]
print('基线示例的标签序列:', [y for _, y in [POOL[i] for i in BASELINE_ORDER]])

print(f"\n{'模型':<20}{'end2end':>9}{'解析率':>9}{'预测分布':>44}")
scores, dists = {}, {}
for m in MODELS:
    b = make_bundle(m, demo_order=BASELINE_ORDER)
    r = run_bundle(b)
    scores[m], dists[m] = r['end_to_end'], r['pred_dist']
    print(f'{m:<20}{r["end_to_end"]:>9.0%}{r["parse_rate"]:>9.0%}'
          f'{str(r["pred_dist"]):>44}')

spread = max(scores.values()) - min(scores.values())
print(f'\n一个**没有针对任何模型调过**的 prompt，四个模型上的跨度: {spread:.0%}')

assert all(v == 1.0 for v in
           [run_bundle(make_bundle(m, BASELINE_ORDER))['parse_rate'] for m in MODELS]), \
    '解码约束让解析率在所有模型上都是 1.0'
assert spread == 0.0, f'基线 prompt 在四个模型上分数相同，实际跨度 {spread}'
# 但预测分布不同
uniq = {frozenset(d.items()) for d in dists.values()}
assert len(uniq) > 1, '预测分布必须不同——行为差异在这里，不在总分上'
print('\n✅ 两个观察，第二个是本模块最重要的伏笔：')
print(f'   ① 解析率在四个模型上都是 1.0——**因为它由解码约束保证，与模型行为无关**。')
print('      这就是「可迁移部分」的价值（讲解第 2 节那张表的上半部分）。')
print(f'   ② 端到端分数**完全相同**（都是 {list(scores.values())[0]:.0%}），')
print('      但**预测分布明显不同**：')
for m in MODELS:
    print(f'      {m:<20}{str(dists[m])}')
print('      也就是说：这四个模型的**行为**确实不同，')
print('      只是在这个没调过的 prompt 上，行为差异恰好没有转化成分数差异。')
print()
print('   下一节会说明这个伏笔的意思：**迁移风险不来自「换模型」本身，')
print('   而来自「这个 prompt 已经针对某个模型调过」。**')

In [ ]:
# --- 为 A 搜一个最优示例顺序，然后搬到 B 上 ---
def best_order_for(model_id, n_trials=200, seed=0, base=None):
    """在**同一个示例集合**上搜排列（不改集合，只改顺序）。"""
    base = BASELINE_ORDER if base is None else list(base)
    rng = np.random.default_rng(seed)
    best, best_a = list(base), -1.0
    for _ in range(n_trials):
        order = [base[i] for i in rng.permutation(len(base))]
        a = run_bundle(make_bundle(model_id, demo_order=order))['end_to_end']
        if a > best_a:
            best, best_a = order, a
    return best, best_a

order_A, acc_A = best_order_for('model-a@2026-01', 200, seed=0)
order_B, acc_B = best_order_for('model-b@2026-06', 200, seed=0)
print(f'为 A 搜出的顺序 {order_A} → A 上 {acc_A:.0%}')
print(f'为 B 搜出的顺序 {order_B} → B 上 {acc_B:.0%}')

base_on_B = run_bundle(make_bundle('model-b@2026-06', BASELINE_ORDER))['end_to_end']
A_order_on_B = run_bundle(make_bundle('model-b@2026-06', order_A))['end_to_end']
base_on_A = run_bundle(make_bundle('model-a@2026-01', BASELINE_ORDER))['end_to_end']
B_order_on_A = run_bundle(make_bundle('model-a@2026-01', order_B))['end_to_end']

print(f"\n{'':<28}{'在 A 上':>10}{'在 B 上':>10}")
print(f'{"基线顺序":<28}{base_on_A:>10.0%}{base_on_B:>10.0%}')
print(f'{"为 A 搜出的顺序":<28}{acc_A:>10.0%}{A_order_on_B:>10.0%}')
print(f'{"为 B 搜出的顺序":<28}{B_order_on_A:>10.0%}{acc_B:>10.0%}')

assert acc_A > base_on_A, '为 A 搜的顺序在 A 上更好'
assert acc_B > base_on_B, '为 B 搜的顺序在 B 上更好'
assert A_order_on_B <= base_on_B, \
    f'为 A 优化的顺序搬到 B 上不该更好: {A_order_on_B:.2f} vs 基线 {base_on_B:.2f}'
print(f'\n✅ **为 A 优化的顺序搬到 B 上是 {A_order_on_B:.0%}，'
      f'而 B 的基线是 {base_on_B:.0%}——不升反降，反而掉了 '
      f'{base_on_B - A_order_on_B:.0%}。**')
print(f'   而为 B 重新搜一遍能拿到 {acc_B:.0%}。')
print()
print('   把它和上一节放在一起，得到本模块的核心结论：')
print(f'   **没调过的 prompt 在四个模型上分数完全相同（都 {base_on_B:.0%}）；')
print('   一旦针对某个模型调过，它就变得不可迁移了。**')
print('   换句话说：**迁移风险是优化的副产物，不是换模型本身的代价。**')
print('   这也给出一个反直觉的推论：如果你从未做过 prompt 层优化，')
print('   那么换模型对你来说是安全的——而做过优化之后，每次换模型都要重做那一半。')
print('   机制在模拟器里是显式的：A 的 recency 是 +0.35（近因偏置），')
print('   B 是 −0.30（**首因偏置**）——位置偏置的方向是相反的。')
print('   真实模型上位置偏置的方向也确实不一致，所以这不是人造的。')
print()
print('   工程结论：**换模型时，示例顺序必须重做，不能搬**。')
print('   而签名、取值域、格式、解码约束可以原样保留（上一节验证过解析率仍是 1.0）。')

## 3 · 迁移的分层复核：总分看不出来的那一层

In [ ]:
def layered_compare(b_old, b_new, sigma=0.05):
    r_old, r_new = run_bundle(b_old), run_bundle(b_new)
    rows = []
    for l in LABELS:
        a, bnew = r_old['by_label'][l], r_new['by_label'][l]
        rows.append((l, a, bnew, bnew - a, (bnew < a - 2 * sigma)))
    return r_old, r_new, rows

B_old = make_bundle('model-a@2026-06', demo_order=order_A)
B_new = make_bundle('model-b@2026-06', demo_order=order_A)   # 直接搬过去（错误做法）
r_o, r_n, rows = layered_compare(B_old, B_new)

print(f"{'层':<10}{'旧':>8}{'新':>8}{'变化':>9}{'显著下降':>10}")
for l, a, bn, d, bad in rows:
    print(f'{l:<10}{a:>8.0%}{bn:>8.0%}{d:>+9.0%}{"是" if bad else "":>10}')
print(f'\n总分: {r_o["end_to_end"]:.0%} → {r_n["end_to_end"]:.0%} '
      f'({r_n["end_to_end"] - r_o["end_to_end"]:+.0%})')

dropped = [l for l, _, _, _, bad in rows if bad]
print(f'显著下降的层: {dropped}')
assert dropped, '直接搬 prompt 过去，至少有一层应当显著下降'
total_drop = r_n['end_to_end'] - r_o['end_to_end']
print(f'\n✅ 有 {len(dropped)} 层显著下降，而总分只变了 {total_drop:+.0%}。')
print('   **门禁必须按层判，不按总分判**——')
print('   这与 C68 模块 03 的切片分析、C70 模块 04 的分层报分是同一条。')

# PSI 是观测量而不是门禁
def psi(p, q, labels=LABELS, eps=1e-6):
    n_p, n_q = sum(p.values()), sum(q.values())
    v = 0.0
    for l in labels:
        a = max(p.get(l, 0) / n_p, eps)
        b_ = max(q.get(l, 0) / n_q, eps)
        v += (a - b_) * math.log(a / b_)
    return v

ps = psi(r_o['pred_dist'], r_n['pred_dist'])
print(f'\n预测分布 PSI = {ps:.3f}')
print('  换模型必然让分布变化，所以 PSI 高**不代表坏**——它是观测量，不设阻断阈值。')
print('  它的用途是：给「新模型的先验与旧模型不同」提供量化证据，据此决定要不要重做校准。')
print('  （与 C70 模块 05 的「top-k 重叠率是观测量不是门禁」完全同构。）')
assert ps > 0.0

## 4 · 成本：token × 单价 × (1 − 缓存命中率)

一个「单价更低但总成本更高」的例子。

In [ ]:
def count_tokens(b):
    text = b['instruction'] + ''.join(a + c for a, c in b['demos'])
    return len(tokenize(text))

def cost_per_request(b, cache_hit=0.0, n_out=8, price_out_ratio=3.0):
    cfg = MODELS[b['model_id']]
    n_in = count_tokens(b) * cfg['token_ratio']
    return (n_in * cfg['price_in'] * (1 - cache_hit)
            + n_out * cfg['price_in'] * price_out_ratio) / 1000.0

print(f"{'模型':<20}{'输入 token':>12}{'单价':>7}{'h=0.9 成本':>12}{'h=0 成本':>11}")
for m in MODELS:
    b = make_bundle(m, BASELINE_ORDER)
    n_in = count_tokens(b) * MODELS[m]['token_ratio']
    print(f'{m:<20}{n_in:>12.0f}{MODELS[m]["price_in"]:>7.1f}'
          f'{cost_per_request(b, 0.9):>12.5f}{cost_per_request(b, 0.0):>11.5f}')

# 「单价更低但总成本更高」
b_a = make_bundle('model-a@2026-06', BASELINE_ORDER)
b_c = make_bundle('model-c@2026-06', BASELINE_ORDER)
print(f'\nmodel-c 的单价是 a 的 {MODELS["model-c@2026-06"]["price_in"]:.1f} 倍，'
      f'但它的 token 数只有 {MODELS["model-c@2026-06"]["token_ratio"]:.2f} 倍')

# 关键对比：示例策略改变缓存命中率
print(f"\n{'示例策略':<22}{'缓存命中率':>12}{'model-b 成本':>14}{'相对固定示例':>14}")
b_b = make_bundle('model-b@2026-06', BASELINE_ORDER)
base_cost = cost_per_request(b_b, cache_hit=0.95)
for name, h in [('固定示例', 0.95), ('分桶 kNN（8 桶）', 0.80), ('全局 kNN', 0.0)]:
    c = cost_per_request(b_b, cache_hit=h)
    print(f'{name:<22}{h:>12.2f}{c:>14.5f}{c / base_cost:>14.2f}×')

c_fixed = cost_per_request(b_b, 0.95)
c_knn = cost_per_request(b_b, 0.0)
assert c_knn > 2 * c_fixed, '全局 kNN 的成本应当明显更高'
print(f'\n✅ 只改示例策略（不改模型、不改 prompt 长度），成本涨了 '
      f'{c_knn / c_fixed:.1f} 倍。')
print('   而模块 02 量到全局 kNN 的准确率收益也很大——**这是一个真实的权衡**，')
print('   分桶 kNN 存在的理由就是它。')
print()
print('   另一条与迁移相关的：**同一段中文 prompt 在不同 tokenizer 下 token 数差 30%+**，')
print('   所以「换模型省钱」必须按 token 数重算，不能只看单价。')

## 5 · 灰度：某一层塌了而总分完全没动

In [ ]:
def canary_metrics(b_old, b_new, share_new=0.10, test=None):
    """灰度：按稳定 ID 分流（不是按请求随机）。返回分桶 × 分层的指标。"""
    test = TEST if test is None else test
    buckets = {'old': [], 'new': []}
    for x, gold in test:
        h = int(hashlib.md5(x.encode()).hexdigest(), 16) % 1000
        buckets['new' if h < share_new * 1000 else 'old'].append((x, gold))
    out = {}
    for name, items in buckets.items():
        if not items:
            out[name] = None
            continue
        out[name] = run_bundle(b_old if name == 'old' else b_new, test=items)
        out[name]['n'] = len(items)
    return out

# 造一个「某一层塌了」的新配置：把 other 类的示例全换成 billing
def sabotage_other(k=8):
    demos = [POOL[i] for i in BASELINE_ORDER]
    return [(t, 'billing' if y == 'other' else y) for t, y in demos]

B_canary = make_bundle('model-a@2026-06', BASELINE_ORDER)
B_canary['demos'] = sabotage_other()

# 用 50% 分流让两桶都有足够样本
m = canary_metrics(make_bundle('model-a@2026-06', BASELINE_ORDER), B_canary,
                   share_new=0.5)
print(f"{'':<14}{'旧桶':>10}{'新桶':>10}")
print(f'{"样本数":<14}{m["old"]["n"]:>10}{m["new"]["n"]:>10}')
print(f'{"parse_rate":<14}{m["old"]["parse_rate"]:>10.0%}'
      f'{m["new"]["parse_rate"]:>10.0%}')
print(f'{"end_to_end":<14}{m["old"]["end_to_end"]:>10.0%}'
      f'{m["new"]["end_to_end"]:>10.0%}')
print('by_label:')
for l in LABELS:
    a = m['old']['by_label'].get(l)
    b_ = m['new']['by_label'].get(l)
    fa = f'{a:.0%}' if a is not None else '—'
    fb = f'{b_:.0%}' if b_ is not None else '—'
    print(f'  {l:<12}{fa:>10}{fb:>10}')

# 加权总分：新桶只占一小部分时，总分几乎不动
for share in (0.10, 0.5):
    mm = canary_metrics(make_bundle('model-a@2026-06', BASELINE_ORDER), B_canary,
                        share_new=share)
    if mm['new'] is None:
        continue
    n_o, n_n = mm['old']['n'], mm['new']['n']
    blended = (mm['old']['end_to_end'] * n_o + mm['new']['end_to_end'] * n_n) / (n_o + n_n)
    pure_old = run_bundle(make_bundle('model-a@2026-06', BASELINE_ORDER))['end_to_end']
    print(f'\n灰度 {share:.0%}: 加权总分 {blended:.0%} vs 全旧配置 {pure_old:.0%} '
          f'(差 {blended - pure_old:+.0%})')

m10 = canary_metrics(make_bundle('model-a@2026-06', BASELINE_ORDER), B_canary,
                     share_new=0.10)
if m10['new'] is not None:
    n_o, n_n = m10['old']['n'], m10['new']['n']
    blended10 = (m10['old']['end_to_end'] * n_o + m10['new']['end_to_end'] * n_n) / (n_o + n_n)
    pure = run_bundle(make_bundle('model-a@2026-06', BASELINE_ORDER))['end_to_end']
    assert abs(blended10 - pure) < 0.15, '小比例灰度时总分几乎不动'
print('\n✅ 三条灰度纪律：')
print('   ① **分流按稳定 ID，不按请求随机**——否则同一用户在同一会话里遇到两套 prompt；')
print('   ② 指标必须按**分桶 × 分层**同时切，总分会把新配置的问题平均掉；')
print('   ③ 两套配置的指纹都要出现在每一条日志里，否则指标无法归因（C68-00 可归因性）。')

## 6 · 监控四项：修复率是解析率的前导指标

In [ ]:
def simulate_drift(model_id, weeks=8, base_compliance=0.98, decay=0.02,
                   constrained=False):
    """模拟格式依从度缓慢下降，看四项指标谁先动。"""
    rows = []
    for w in range(weeks):
        comp = base_compliance - decay * w
        n = 400
        rng = np.random.default_rng(w)
        raw_ok = rng.random(n) < comp                     # 输出本身合格
        if constrained:
            parse_ok = np.ones(n, dtype=bool)             # 约束保证
            repaired = np.zeros(n, dtype=bool)
        else:
            # 不合格的里面，语法层修复能救回 70%
            rescued = (~raw_ok) & (rng.random(n) < 0.7)
            parse_ok = raw_ok | rescued
            repaired = rescued
        rows.append(dict(week=w, compliance=comp,
                         parse_rate=float(parse_ok.mean()),
                         repair_rate=float(repaired.mean())))
    return rows

print('不带解码约束（格式依从度每周降 2 个点）:')
print(f"{'week':>5}{'依从度':>9}{'解析率':>9}{'修复率':>9}")
rows = simulate_drift('model-a@2026-06', constrained=False)
for r in rows:
    print(f'{r["week"]:>5}{r["compliance"]:>9.2f}{r["parse_rate"]:>9.3f}'
          f'{r["repair_rate"]:>9.3f}')

# 修复率的变化幅度大于解析率的变化幅度 → 它更灵敏
d_parse = rows[0]['parse_rate'] - rows[-1]['parse_rate']
d_repair = rows[-1]['repair_rate'] - rows[0]['repair_rate']
print(f'\n8 周里：解析率下降 {d_parse:.3f}，修复率上升 {d_repair:.3f}')
assert d_repair > d_parse, '修复率的变化幅度更大 → 它是更灵敏的信号'

rows_c = simulate_drift('model-a@2026-06', constrained=True)
assert all(r['parse_rate'] == 1.0 for r in rows_c), '约束下解析率恒为 1.0'
assert all(r['repair_rate'] == 0.0 for r in rows_c)
print('✅ 带解码约束时解析率恒为 1.0、修复率恒为 0——')
print('   **此时「解析率 < 1.0」直接是实现 bug 的信号**（模块 04 练习 4）。')
print(f'   而不带约束时，修复率的变化幅度是解析率的 {d_repair / d_parse:.1f} 倍——')
print('   **修复率是解析率的前导指标**。而它只在修复被显式记录时才存在。')

In [ ]:
# --- 第四项：无内容探针 ---
def probe(model_id, bundle, probes=('N/A', '', '[MASK]')):
    return Counter(toy_lm(model_id, bundle['instruction'], bundle['demos'], p)
                   for p in probes)

print(f"{'模型':<20}{'探针输出':>26}")
priors = {}
for m in MODELS:
    b = make_bundle(m, BASELINE_ORDER)
    pr = probe(m, b)
    priors[m] = set(pr)
    print(f'{m:<20}{str(dict(pr)):>26}')

assert len({frozenset(v) for v in priors.values()}) > 1, '不同模型的先验不同'
print('\n✅ 无内容探针每次一次调用，就把「模型的默认倾向」变成一个可比的字符串。')
print('   它变化时，三种原因都值得知道：模型换版本 / 示例池变了 / 有人改了顺序——')
print('   **而三种都不会在准确率上立刻显现。**')
print()
print('   四项监控里有三项（修复率、预测分布、探针）都不是准确率。')
print('   这正是本模块的要点：**prompt 层的问题在准确率上反应很慢，')
print('   在这三项上反应很快。**')

## 7 · prompt 运维卡 + 门禁

In [ ]:
def prompt_fp(b):
    payload = dict(signature=b['signature'], instruction=b['instruction'],
                   demos=[[a, c] for a, c in b['demos']],
                   format_spec=b['format_spec'], decoding=b['decoding'],
                   model_id=b['model_id'])
    return hashlib.sha256(json.dumps(payload, sort_keys=True,
                                     ensure_ascii=False).encode()).hexdigest()[:12]

def ops_card(b, metrics, baseline_metrics, changelog, rollback_to,
             cache_hit=0.8, sigma=0.05):
    ps = psi(baseline_metrics['pred_dist'], metrics['pred_dist'])
    return dict(
        model_id=b['model_id'], prompt_fp=prompt_fp(b), changelog=changelog,
        constrained=b['decoding']['constrained'],
        parse_rate=metrics['parse_rate'], repair_rate=metrics['repair_rate'],
        by_label=metrics['by_label'],
        baseline_by_label=baseline_metrics['by_label'],
        psi=ps, probe=sorted(probe(b['model_id'], b)),
        baseline_probe=sorted(probe('model-a@2026-01',
                                    make_bundle('model-a@2026-01', BASELINE_ORDER))),
        cost=cost_per_request(b, cache_hit=cache_hit),
        cache_hit=cache_hit, rollback_to=rollback_to, sigma=sigma)

def ops_gate(card, prev_changelog=None):
    blocking, warn, observe = [], [], []
    # 确定性
    if 'latest' in card['model_id'] or '@' not in card['model_id']:
        blocking.append(f"模型 ID {card['model_id']!r} 未钉死版本")
    if prev_changelog is not None and card['changelog'] == prev_changelog:
        blocking.append('指纹变了但 CHANGELOG 没动')
    if card['constrained'] and card['parse_rate'] < 1.0:
        blocking.append(f"启用约束但解析率 {card['parse_rate']:.2f} < 1.0（实现有 bug）")
    if not card['rollback_to']:
        blocking.append('缺回滚指针——这次变更不可回滚，不该上线')
    # 统计：按层判
    for l, v in card['by_label'].items():
        b0 = card['baseline_by_label'].get(l)
        if b0 is not None and v < b0 - 2 * card['sigma']:
            blocking.append(f'{l} 层准确率下降 {b0:.2f} → {v:.2f}（> 2σ）')
    # 报警
    if card['repair_rate'] > 0:
        warn.append(f"修复率 {card['repair_rate']:.1%} > 0（解析率的前导指标）")
    if card['probe'] != card['baseline_probe']:
        warn.append(f"探针输出变了: {card['baseline_probe']} → {card['probe']}")
    # 观测（不设阈值）
    observe.append(f"预测分布 PSI {card['psi']:.3f}（换模型必然变，不设阻断）")
    return blocking, warn, observe

BASE_M = run_bundle(make_bundle('model-a@2026-01', BASELINE_ORDER))
GOOD = make_bundle('model-b@2026-06', demo_order=order_B)     # 为 B 重做了顺序
card_good = ops_card(GOOD, run_bundle(GOOD), BASE_M,
                     'v7: 迁移到 model-b，重做示例顺序与校准', 'v6')
b, w, o = ops_gate(card_good, prev_changelog='v6: 初版')
print('为 B 重做顺序后:')
print('  阻断:', b if b else '无')
print('  报警:', w if w else '无')
print('  观测:', o)

BAD = make_bundle('model-b@2026-06', demo_order=order_A)      # 直接搬 A 的顺序
card_bad = ops_card(BAD, run_bundle(BAD), BASE_M,
                    'v7: 迁移到 model-b（直接搬旧配置）', 'v6')
b2, _, _ = ops_gate(card_bad, prev_changelog='v6: 初版')
print(f'\n直接搬 A 的顺序: 阻断 {len(b2)} 项')
for x in b2[:3]:
    print('  ·', x)

assert any('层准确率下降' in x for x in b2), b2
# 模型 ID 没钉版本 → 阻断
card_latest = dict(card_good); card_latest['model_id'] = 'model-b:latest'
assert any('未钉死版本' in x for x in ops_gate(card_latest)[0])
# 缺回滚指针 → 阻断
card_norb = dict(card_good); card_norb['rollback_to'] = None
assert any('回滚' in x for x in ops_gate(card_norb)[0])
# CHANGELOG 没动 → 阻断
assert any('CHANGELOG' in x for x in
           ops_gate(card_good, prev_changelog=card_good['changelog'])[0])
print('\n✅ 四项确定性阻断 + 按层的统计阻断 + 两项报警 + 一项观测。')
print('   而这张卡里最重要的一行仍然是回滚指针——与 C70 模块 05 完全一致：')
print('   **如果回滚需要手动编辑 prompt，那么这次变更是不可回滚的，不该上线。**')

## ✏️ 练习 1：可迁移性分类器

实现 `migration_plan(bundle)`：把 bundle 的每一部分归到
`'portable'`（跨模型保留）/ `'retest'`（保留但必须重测）/ `'redo'`（必须重做）。

依据讲解第 2 节那张表：
- `portable`：`signature` / `domain`（取值域）/ `decoding`
- `retest`：`format_spec`（保留但解析率必须重测）
- `redo`：`demos`（顺序）/ `instruction`（措辞）/ `calib`

返回 `dict(portable=[...], retest=[...], redo=[...])`，键名排序。
再实现 `migration_effort(plan)`：返回 `len(redo) / (总项数)`——
**这个比例就是「换模型要重做多少」。**

In [ ]:
def migration_plan(bundle):
    """返回 dict(portable=[...], retest=[...], redo=[...])，每个列表已排序。"""
    # TODO
    raise NotImplementedError

def migration_effort(plan):
    """返回 redo 占全部项的比例。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
plan = migration_plan(B_A)
print(plan)
assert set(plan) == {'portable', 'retest', 'redo'}
assert 'signature' in plan['portable'] and 'decoding' in plan['portable']
assert 'domain' in plan['portable']
assert plan['retest'] == ['format_spec'], plan['retest']
assert set(plan['redo']) == {'calib', 'demos', 'instruction'}, plan['redo']
for k in plan:
    assert plan[k] == sorted(plan[k]), f'{k} 未排序'

eff = migration_effort(plan)
print(f'换模型要重做的比例: {eff:.0%}')
assert abs(eff - 3 / 7) < 1e-9, eff
# 如果不用解码约束（把它从 portable 里去掉），重做比例会更高
plan2 = migration_plan({**B_A, 'decoding': dict(constrained=False)})
assert 'decoding' in plan2['portable'], '不管开不开，decoding 本身都是可迁移的配置项'
print('✅ 练习 1 通过：把 bundle 分成三档，「换模型」就变成「换那一档」')

## 📖 参考答案 1

In [ ]:
# 练习 1 参考答案
PORTABLE_KEYS = ('signature', 'domain', 'decoding')
RETEST_KEYS = ('format_spec',)
REDO_KEYS = ('demos', 'instruction', 'calib')

def migration_plan(bundle):
    out = dict(portable=[], retest=[], redo=[])
    for k in PORTABLE_KEYS:
        if k == 'domain':
            if 'signature' in bundle and 'domain' in bundle['signature']:
                out['portable'].append('domain')
        elif k in bundle:
            out['portable'].append(k)
    for k in RETEST_KEYS:
        if k in bundle:
            out['retest'].append(k)
    for k in REDO_KEYS:
        if k in bundle:
            out['redo'].append(k)
    return {k: sorted(v) for k, v in out.items()}

def migration_effort(plan):
    total = sum(len(v) for v in plan.values())
    return len(plan['redo']) / total if total else 0.0

plan = migration_plan(B_A)
assert plan['retest'] == ['format_spec']
assert set(plan['redo']) == {'calib', 'demos', 'instruction'}
assert abs(migration_effort(plan) - 3 / 7) < 1e-9
print('✅ 参考答案 1 通过')
print(f'   本例的重做比例是 {migration_effort(plan):.0%}——三项里的三项。')
print('   而这个比例是**可以设计的**：')
print('   把格式保证从「指令措辞」搬到「解码约束」，就把一项从 redo 挪到了 portable。')
print('   这是模块 04 那句「约束在长期上比指令优化更值」的量化形式。')

## ✏️ 练习 2：迁移决策

实现 `migration_decision(old_bundle, new_model, sigma=0.05, n_trials=200)`：

1. 直接搬（`transfer`）：把旧顺序搬到新模型
2. 重做（`redo`）：为新模型搜一个最优顺序
3. 对每种方案跑分层比较，返回

```
dict(baseline_new=..., transfer=..., redo=...,
     transfer_layers_dropped=[...], redo_layers_dropped=[...],
     recommendation='redo'|'transfer'|'stay')
```

推荐规则：
- `redo` 无层下降且 `redo` 总分 ≥ 旧模型总分 → `'redo'`
- 否则若 `transfer` 无层下降且总分 ≥ 旧模型总分 → `'transfer'`
- 否则 → `'stay'`（不迁移）

In [ ]:
def migration_decision(old_bundle, new_model, sigma=0.05, n_trials=200):
    """返回上面描述的 dict。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
OLD = make_bundle('model-a@2026-06', demo_order=order_A)
d = migration_decision(OLD, 'model-b@2026-06')
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in d.items()})

assert set(d) >= {'baseline_new', 'transfer', 'redo', 'transfer_layers_dropped',
                  'redo_layers_dropped', 'recommendation'}
assert d['recommendation'] in ('redo', 'transfer', 'stay')
# 重做至少不比直接搬差
assert d['redo'] >= d['transfer'] - 1e-9, (d['redo'], d['transfer'])
# 直接搬会掉层，而重做掉的层更少
assert len(d['transfer_layers_dropped']) >= len(d['redo_layers_dropped'])

# 迁到同家族的小版本：变化应当更小
d2 = migration_decision(make_bundle('model-a@2026-01', demo_order=order_A),
                        'model-a@2026-06')
print('\n迁到同家族小版本:', {k: (round(v, 3) if isinstance(v, float) else v)
                        for k, v in d2.items()})
assert len(d2['transfer_layers_dropped']) <= len(d['transfer_layers_dropped']), \
    '同家族迁移的层级损失不该比跨家族更大'
print('✅ 练习 2 通过：迁移决策由分层结果给出，而不是由总分给出')

## 📖 参考答案 2

In [ ]:
# 练习 2 参考答案
def migration_decision(old_bundle, new_model, sigma=0.05, n_trials=200):
    old_m = run_bundle(old_bundle)
    old_order = [POOL.index(d) for d in old_bundle['demos']]

    b_transfer = make_bundle(new_model, demo_order=old_order)
    m_transfer = run_bundle(b_transfer)

    order_new, _ = best_order_for(new_model, n_trials=n_trials, seed=0,
                                  base=old_order)
    b_redo = make_bundle(new_model, demo_order=order_new)
    m_redo = run_bundle(b_redo)

    def dropped(m):
        return sorted(l for l in LABELS
                      if m['by_label'].get(l, 0.0)
                      < old_m['by_label'].get(l, 0.0) - 2 * sigma)

    d_t, d_r = dropped(m_transfer), dropped(m_redo)
    if not d_r and m_redo['end_to_end'] >= old_m['end_to_end'] - 1e-9:
        rec = 'redo'
    elif not d_t and m_transfer['end_to_end'] >= old_m['end_to_end'] - 1e-9:
        rec = 'transfer'
    else:
        rec = 'stay'
    return dict(baseline_new=run_bundle(make_bundle(new_model,
                                                    BASELINE_ORDER))['end_to_end'],
                transfer=m_transfer['end_to_end'], redo=m_redo['end_to_end'],
                transfer_layers_dropped=d_t, redo_layers_dropped=d_r,
                recommendation=rec)

d = migration_decision(OLD, 'model-b@2026-06')
assert d['recommendation'] in ('redo', 'transfer', 'stay')
assert d['redo'] >= d['transfer'] - 1e-9
assert len(d['transfer_layers_dropped']) >= len(d['redo_layers_dropped'])
print('✅ 参考答案 2 通过')
print(f"   本例推荐: {d['recommendation']}")
print('   注意 `stay` 这个选项的存在很重要：**「迁移」不是一个必然要完成的动作**。')
print('   如果重做之后仍有层下降，正确的结论是「先不迁」，')
print('   而不是「迁了再想办法」——因为迁移之后你已经失去了对照。')

## ✏️ 练习 3：总成本比较（含 tokenizer 与缓存）

实现 `total_cost_compare(models, demo_strategy_hit, n_out=8, qps_hours=1.0)`：
对每个模型算出

```
dict(model_id -> dict(n_in_tokens, price_in, cache_hit, cost_per_req, cost_per_hour))
```

`demo_strategy_hit` 是示例策略对应的缓存命中率。
`cost_per_hour = cost_per_req * qps_hours * 3600`。

然后实现 `cheapest(models, hit)`：返回按 `cost_per_req` 最便宜的 model_id。

用它验证一个反直觉的结论：**单价最低的模型不一定总成本最低。**

In [ ]:
def total_cost_compare(models, demo_strategy_hit, n_out=8, qps_hours=1.0):
    """返回 {model_id: dict(...)}。"""
    # TODO
    raise NotImplementedError

def cheapest(models, hit):
    """返回 cost_per_req 最低的 model_id。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
tbl = total_cost_compare(list(MODELS), demo_strategy_hit=0.9)
print(f"{'模型':<20}{'输入 token':>12}{'单价':>7}{'成本/请求':>12}{'成本/小时':>12}")
for m, v in tbl.items():
    print(f'{m:<20}{v["n_in_tokens"]:>12.0f}{v["price_in"]:>7.1f}'
          f'{v["cost_per_req"]:>12.5f}{v["cost_per_hour"]:>12.2f}')

assert set(tbl) == set(MODELS)
for v in tbl.values():
    assert v['cache_hit'] == 0.9
    assert abs(v['cost_per_hour'] - v['cost_per_req'] * 3600) < 1e-9

# 单价最低的模型
cheapest_price = min(MODELS, key=lambda m: MODELS[m]['price_in'])
print(f'\n单价最低的模型: {cheapest_price} '
      f'(price {MODELS[cheapest_price]["price_in"]:.1f})')
print(f'总成本最低的模型: {cheapest(list(MODELS), 0.9)}')

# 缓存命中率会改变排序
c_high = cheapest(list(MODELS), 0.95)
c_low = cheapest(list(MODELS), 0.0)
print(f'h=0.95 时最便宜: {c_high} | h=0 时最便宜: {c_low}')
# token_ratio 让「单价低」不等于「总成本低」
b_b = make_bundle('model-b@2026-06', BASELINE_ORDER)
b_c = make_bundle('model-c@2026-06', BASELINE_ORDER)
assert MODELS['model-b@2026-06']['token_ratio'] > \
       MODELS['model-c@2026-06']['token_ratio'], 'b 的 token 数更多'
print('✅ 练习 3 通过：总成本要按「token 数 × 单价 × (1−缓存命中率)」算')

## 📖 参考答案 3

In [ ]:
# 练习 3 参考答案
def total_cost_compare(models, demo_strategy_hit, n_out=8, qps_hours=1.0):
    out = {}
    for m in models:
        b = make_bundle(m, BASELINE_ORDER)
        cfg = MODELS[m]
        n_in = count_tokens(b) * cfg['token_ratio']
        cpr = cost_per_request(b, cache_hit=demo_strategy_hit, n_out=n_out)
        out[m] = dict(n_in_tokens=n_in, price_in=cfg['price_in'],
                      cache_hit=demo_strategy_hit, cost_per_req=cpr,
                      cost_per_hour=cpr * qps_hours * 3600)
    return out

def cheapest(models, hit):
    tbl = total_cost_compare(models, hit)
    return min(tbl, key=lambda m: tbl[m]['cost_per_req'])

tbl = total_cost_compare(list(MODELS), 0.9)
assert set(tbl) == set(MODELS)
for v in tbl.values():
    assert abs(v['cost_per_hour'] - v['cost_per_req'] * 3600) < 1e-9
print('✅ 参考答案 3 通过')
print('   两个容易被忽略的乘数：')
print('   ① **tokenizer**：同一段中文 prompt 在不同 tokenizer 下 token 数差 30%+，')
print('      所以「单价低」不等于「总成本低」；')
print('   ② **缓存命中率**：它由示例策略决定（模块 02 第 3 节的分桶 kNN），')
print('      而它出现在成本公式的括号里——全局 kNN 会把这一项直接归零。')
print('   而输出 token 的单价通常是输入的几倍，所以「让模型少说话」')
print('   （模块 01 的禁止项槽位 + 模块 04 的解码约束）也是一项成本优化。')

## ✏️ 练习 4：完整的运维门禁

把第 7 节的门禁补全为 `full_gate(card, prev_card)`，在原有规则之外再加三条：

**确定性阻断**
- `card['prompt_fp'] == prev_card['prompt_fp']` 但 changelog 不同
  （「改了说明但什么都没改」）
- `card['constrained']` 为假而 `card['parse_rate'] < 1.0` **且**
  `card['repair_rate'] == 0`（没开约束、解析失败、又没有修复记录 → 修复是静默的）

**报警**
- `card['cost'] > prev_card['cost'] * 1.2`（成本上升超过 20%）

返回 `(blocking, warnings, observations)`。

In [ ]:
def full_gate(card, prev_card):
    """返回 (blocking, warnings, observations)。"""
    # TODO：先调用 ops_gate，再加三条
    raise NotImplementedError

In [ ]:
# —— 自测 ——
PREV = ops_card(make_bundle('model-a@2026-01', BASELINE_ORDER),
                BASE_M, BASE_M, 'v6: 初版', 'v5')

b, w, o = full_gate(card_good, PREV)
print('正常迁移: 阻断', len(b), '| 报警', len(w), '| 观测', len(o))
assert not any('什么都没改' in x for x in b)

# a) 指纹相同但 changelog 不同
same_fp = dict(card_good); same_fp['prompt_fp'] = PREV['prompt_fp']
b2, _, _ = full_gate(same_fp, PREV)
assert any('什么都没改' in x for x in b2), b2

# b) 没开约束 + 解析失败 + 无修复记录 → 静默修复
silent = dict(card_good)
silent['constrained'] = False
silent['parse_rate'] = 0.93
silent['repair_rate'] = 0.0
b3, _, _ = full_gate(silent, PREV)
assert any('静默' in x for x in b3), b3

# c) 成本上升超过 20% → 报警
pricey = dict(card_good); pricey['cost'] = PREV['cost'] * 1.5
_, w4, _ = full_gate(pricey, PREV)
assert any('成本' in x for x in w4), w4

# d) 原有规则仍然生效
nolatest = dict(card_good); nolatest['model_id'] = 'model-b:latest'
assert any('未钉死版本' in x for x in full_gate(nolatest, PREV)[0])
print('✅ 练习 4 通过：七项确定性阻断 + 按层统计阻断 + 三项报警 + 一项观测')

## 📖 参考答案 4

In [ ]:
# 练习 4 参考答案
def full_gate(card, prev_card):
    blocking, warn, observe = ops_gate(card, prev_changelog=prev_card['changelog'])
    if card['prompt_fp'] == prev_card['prompt_fp'] and \
            card['changelog'] != prev_card['changelog']:
        blocking.append('指纹与上一版相同但 changelog 不同——改了说明但什么都没改')
    if (not card['constrained']) and card['parse_rate'] < 1.0 \
            and card['repair_rate'] == 0:
        blocking.append('没开约束、解析率 < 1.0、却没有任何修复记录'
                        '——修复是静默的，信号被丢掉了')
    if card['cost'] > prev_card['cost'] * 1.2:
        warn.append(f"成本上升 {card['cost'] / prev_card['cost']:.2f}× > 1.2×")
    return blocking, warn, observe

b, w, o = full_gate(card_good, PREV)
same_fp = {**card_good, 'prompt_fp': PREV['prompt_fp']}
assert any('什么都没改' in x for x in full_gate(same_fp, PREV)[0])
silent = {**card_good, 'constrained': False, 'parse_rate': 0.93, 'repair_rate': 0.0}
assert any('静默' in x for x in full_gate(silent, PREV)[0])
assert any('成本' in x for x in
           full_gate({**card_good, 'cost': PREV['cost'] * 1.5}, PREV)[1])
print('✅ 参考答案 4 通过')
print('   第二条新增检查值得单独说：')
print('   **「解析率 < 1.0 而修复率 = 0」是一个自相矛盾的组合**——')
print('   要么修复其实发生了但没被记录（信号丢了），')
print('   要么解析失败被静默当成了「答错」（模块 01 的三个量被混成了一个）。')
print('   两种情形都需要人看，所以它是阻断而不是报警。')
print()
print('   而这套门禁里最重要的一条仍然是回滚指针（第 7 节）：')
print('   **如果回滚需要手动编辑 prompt，那么这次变更是不可回滚的，不该上线。**')

## 🧪 真实工程胶囊：prompt 运维的落地

```python
# ══════════════════════════════════════════════════════════════════
# A. 目录结构：可迁移与不可迁移分开存（讲解第 2 节 / 练习 1）
# ══════════════════════════════════════════════════════════════════
prompts/classify_feedback/
  portable/                       # ← 与模型无关，一份
    signature.json                #   inputs / output / domain / invariants
    format.json
    decoding.json                 #   JSON schema + 约束开关
  models/
    model-a@2026-06/
      instruction.txt  demos.jsonl  calib.json  CHANGELOG.md
    model-b@2026-06/
      instruction.txt  demos.jsonl  calib.json  CHANGELOG.md
  ACTIVE -> models/model-a@2026-06     # ← 一个指针；回滚 = 反向切它

# ══════════════════════════════════════════════════════════════════
# B. 模型 ID 必须钉死（讲解第 1 节）
# ══════════════════════════════════════════════════════════════════
MODEL = 'claude-x@2026-06-01'     # ← 不是 'latest'、不是不带版本
#   钉死之后，服务方更新不会静默改变你的 prompt 面对的对象。
#   而它进指纹，所以「换模型」必然产生一次可见的变更记录。

# ══════════════════════════════════════════════════════════════════
# C. 迁移流程（讲解第 3 节，结构直接来自 C70 模块 05）
# ══════════════════════════════════════════════════════════════════
# 1) 影子评测：(bundle_A, model_B) 与 (bundle_B*, model_B) 都跑
# 2) 三条校验：解析率（确定性）· 分层准确率（统计，按层判）· PSI（观测，不设阈值）
# 3) 灰度 1% → 10% → 50% → 100%，**分流按稳定 ID 哈希，不按请求随机**
# 4) 每一档看分桶 × 分层的指标；两套配置的指纹都进日志
# 5) 保留旧配置 N 天；回滚 = 反向切 ACTIVE

# ══════════════════════════════════════════════════════════════════
# D. 监控四项（讲解第 6 节）—— 三项都不是准确率
# ══════════════════════════════════════════════════════════════════
# parse_rate                    约束下恒为 1.0；掉下来 = 实现 bug
# repair_rate / retry_rate      **解析率的前导指标**；必须显式记录修复
# pred_label_distribution + PSI  抓顺序偏置 / 示例漂移 / 模型换版本
# content_free_probe            每小时一次调用，跟踪标签先验

# ══════════════════════════════════════════════════════════════════
# E. 成本（讲解第 4 节 / 练习 3）
# ══════════════════════════════════════════════════════════════════
# cost = (n_instr + Σ n_demo + n_x) · price_in · (1 − cache_hit) + n_out · price_out
#   两个易忘的乘数：tokenizer（中文差 30%+）与 cache_hit（由示例策略决定）
#   换模型「省钱」必须按 token 数重算，不能只看单价。
```

---

## 小结

| 结论 | 数字 | 在哪一节 |
|---|---|---|
| prompt 是会随模型失效的资产 | 四种失效，三种只能靠监控发现 | 讲解 1 |
| 签名/取值域/格式/解码约束跨模型保留 | 解析率在四个模型上都是 1.0 | 第 2 节 |
| **为 A 优化的示例顺序搬到 B 上不升反降** | 而为 B 重做能拿到更高分 | 第 2 节 |
| 位置偏置的方向在不同模型上可以相反 | A 是 +0.35（近因），B 是 −0.30（首因） | 第 2 节 |
| 迁移的门禁必须按层判，不按总分判 | 有层显著下降而总分几乎没动 | 第 3 节 |
| PSI 是观测量不是门禁 | 换模型必然让分布变化 | 第 3 节 |
| 只改示例策略，成本涨几倍 | 缓存命中率 0.95 → 0 | 第 4 节 |
| 单价最低不等于总成本最低 | tokenizer 差 30%+ | 第 4 节 / 练习 3 |
| 灰度分流按稳定 ID，不按请求随机 | 否则同一会话遇到两套 prompt | 第 5 节 |
| 修复率是解析率的前导指标 | 8 周里它的变化幅度更大 | 第 6 节 |
| 「解析率 < 1.0 而修复率 = 0」自相矛盾 | 修复是静默的，信号丢了 | 练习 4 |
| 回滚必须是切指针 | 需要手动编辑 = 不可回滚 = 不该上线 | 第 7 节 |

**C71 完结。** 全课的一句话：
**prompt 是一个有契约、可分层测量、可搜索、可保证、需要运维的程序——
而这五件事的顺序不能换：先定契约（01）、再拿免费收益（02）、
再自动搜索（03）、能约束的用约束（04）、最后把它当资产管起来（05）。**